In [3]:
import requests
import time
import random
import pandas as pd
import duckdb

DB_PATH = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb"

con = duckdb.connect(DB_PATH)
print("Connected:", DB_PATH)

pd.options.mode.chained_assignment = None
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


Connected: C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb


# SINGLE SEASON 

In [2]:



# ================================
# SAFE REQUEST (429 BACKOFF)
# ================================
def safe_request(url, params, max_retries=5):
    backoff = 1.0
    for attempt in range(max_retries):
        resp = requests.get(url, params=params)

        if resp.status_code == 200:
            return resp

        if resp.status_code == 429:
            print(f"⚠️ 429 rate limit. Waiting {backoff:.1f}s...")
            time.sleep(backoff)
            backoff *= 2
            continue

        print(f"⚠️ Error {resp.status_code}. Retrying {backoff:.1f}s...")
        time.sleep(backoff)
        backoff *= 2

    raise Exception(f"Failed after {max_retries} retries (status {resp.status_code})")

# ================================
# GOALIE SCRAPER WITH TOTAL PAGINATION
# ================================
def get_goalie_games(start_date, end_date, base_delay=1.0):
    url = "https://api.nhle.com/stats/rest/en/goalie/summary"
    limit = 100  # API always uses 100 rows per page

    cayenne = (
        f'gameDate<="{end_date} 23:59:59" '
        f'and gameDate>="{start_date}"'
    )

    params = {
        "isAggregate": "false",
        "isGame": "true",
        "sort": '[{"property":"wins","direction":"DESC"},'
                '{"property":"savePct","direction":"DESC"},'
                '{"property":"playerId","direction":"ASC"}]',
        "cayenneExp": cayenne,
        "start": 0,
        "limit": limit
    }

    # ---- Fetch first page to get total ----
    resp = safe_request(url, params)
    payload = resp.json()

    if "total" not in payload:
        raise Exception("API missing 'total' — cannot paginate goalie data")

    total = payload["total"]
    print(f"→ API reports total goalie rows = {total}")

    all_rows = []
    start = 0

    # ---- Loop using total ----
    while start < total:
        params["start"] = start
        resp = safe_request(url, params)
        page = resp.json().get("data", [])

        print(f"   Fetch rows {start} → {start+limit} : {len(page)} rows")

        if not page:
            print("⚠️ Empty page before finishing — stopping early.")
            break

        all_rows.extend(page)
        start += limit

        time.sleep(base_delay + random.uniform(0.25, 0.5))

    # Trim excess rows if duplicated
    if len(all_rows) > total:
        all_rows = all_rows[:total]

    print(f"✔ Downloaded {len(all_rows)} goalie rows (expected {total})")
    return pd.DataFrame(all_rows)

# ================================
# RUN SCRAPER FOR CURRENT SEASON
# ================================
today = pd.to_datetime(date.today()).strftime("%Y-%m-%d")
start_date = "2000-10-04" # season start
end_date = "2013-04-27"   # up to today

goalie_df = get_goalie_games(start_date, end_date)

goalie_df.tail()


→ API reports total goalie rows = 10000
   Fetch rows 0 → 100 : 100 rows
   Fetch rows 100 → 200 : 100 rows
   Fetch rows 200 → 300 : 100 rows
   Fetch rows 300 → 400 : 100 rows
   Fetch rows 400 → 500 : 100 rows
   Fetch rows 500 → 600 : 100 rows
   Fetch rows 600 → 700 : 100 rows
   Fetch rows 700 → 800 : 100 rows
   Fetch rows 800 → 900 : 100 rows
   Fetch rows 900 → 1000 : 100 rows
   Fetch rows 1000 → 1100 : 100 rows
   Fetch rows 1100 → 1200 : 100 rows
   Fetch rows 1200 → 1300 : 100 rows
   Fetch rows 1300 → 1400 : 100 rows
   Fetch rows 1400 → 1500 : 100 rows
   Fetch rows 1500 → 1600 : 100 rows


KeyboardInterrupt: 

# Multiple Past Seasons

In [ ]:

# =========================================
# SEASON DATE RANGES
# =========================================
season_start_dates = {
    20002001: "2000-10-04",
    20012002: "2001-10-03",
    20022003: "2002-10-09",
    20032004: "2003-10-08",
    20052006: "2005-10-05",
    20062007: "2006-10-04",
    20072008: "2007-09-29",
    20082009: "2008-10-04",
    20092010: "2009-10-01",
    20102011: "2010-10-07",
    20112012: "2011-10-06",
    20122013: "2013-01-19"
}
    
    
season_end_dates = {
   20002001: "2001-04-08",
    20012002: "2002-04-14",
    20022003: "2003-04-06",
    20032004: "2004-04-04",
    20052006: "2006-04-18",
    20062007: "2007-04-08",
    20072008: "2008-04-06",
    20082009: "2009-04-12",
    20092010: "2010-04-11",
    20102011: "2011-04-10",
    20112012: "2012-04-07",
    20122013: "2013-04-27",
}

# =========================================
# SAFE REQUEST (429 BACKOFF)
# =========================================
def safe_request(url, params, max_retries=5):
    backoff = 1.0
    for attempt in range(max_retries):
        resp = requests.get(url, params=params)

        if resp.status_code == 200:
            return resp

        if resp.status_code == 429:
            print(f"⚠️ 429 rate limit. Waiting {backoff:.1f}s...")
            time.sleep(backoff)
            backoff *= 2
            continue

        print(f"⚠️ Error {resp.status_code}. Retrying in {backoff:.1f}s...")
        time.sleep(backoff)
        backoff *= 2

    raise Exception(f"Failed after {max_retries} retries (status {resp.status_code})")

# =========================================
# GOALIE SCRAPER — FULL PAGINATION
# =========================================
def get_goalie_by_season(start_date, end_date, base_delay=1.0):
    url = "https://api.nhle.com/stats/rest/en/goalie/summary"
    limit = 100

    cayenne = (
        f'gameDate<="{end_date} 23:59:59" '
        f'and gameDate>="{start_date}"'
    )

    params = {
        "isAggregate": "false",
        "isGame": "true",
        "sort": '[{"property":"wins","direction":"DESC"},'
                '{"property":"savePct","direction":"DESC"},'
                '{"property":"playerId","direction":"ASC"}]',
        "cayenneExp": cayenne,
        "start": 0,
        "limit": limit,
    }

    # ---- First call to find total ----
    resp = safe_request(url, params)
    payload = resp.json()

    if "total" not in payload:
        raise Exception("API missing 'total' for pagination")

    total = payload["total"]
    print(f"  → API reports total rows = {total}")

    all_rows = []
    start = 0

    # ---- Loop using total ----
    while start < total:
        params["start"] = start
        resp = safe_request(url, params)
        page = resp.json().get("data", [])

        print(f"     Fetching rows {start} → {start+limit} : {len(page)} rows")

        if not page:
            print("⚠️ Empty page encountered early — stopping.")
            break

        all_rows.extend(page)
        start += limit

        time.sleep(base_delay + random.uniform(0.25, 0.5))

    # ---- Trim extra rows ----
    if len(all_rows) > total:
        all_rows = all_rows[:total]

    return pd.DataFrame(all_rows)

# =========================================
# MULTI-SEASON LOOP
# =========================================
all_seasons_data = []

for season in season_start_dates:
    start_date = season_start_dates[season]
    end_date = season_end_dates[season]

    print(f"\n=== Fetching GOALIE DATA for {season}: {start_date} → {end_date} ===")

    df = get_goalie_by_season(start_date, end_date)

    if not df.empty:
        df["season"] = season
        all_seasons_data.append(df)

# =========================================
# FINAL CLEANUP
# =========================================
final_df = pd.concat(all_seasons_data, ignore_index=True)

goalie_games = final_df[[ 
    'gameId', 'gamesPlayed', 'gamesStarted', 'goalieFullName', 'playerId',
    'teamAbbrev', 'timeOnIce', 'wins', 'losses', 'otLosses',
    'goalsAgainstAverage', 'saves', 'shutouts', 'savePct',
    'shotsAgainst', 'goalsAgainst', 'season'
]]

goalie_games = goalie_games.rename(columns={'gameId':'game_id', 'season':'season_id'})



=== Fetching GOALIE DATA for 20002001: 2000-10-04 → 2001-04-08 ===
  → API reports total rows = 2664
     Fetching rows 0 → 100 : 100 rows
     Fetching rows 100 → 200 : 100 rows
     Fetching rows 200 → 300 : 100 rows
     Fetching rows 300 → 400 : 100 rows
     Fetching rows 400 → 500 : 100 rows
     Fetching rows 500 → 600 : 100 rows
     Fetching rows 600 → 700 : 100 rows
     Fetching rows 700 → 800 : 100 rows
     Fetching rows 800 → 900 : 100 rows
     Fetching rows 900 → 1000 : 100 rows
     Fetching rows 1000 → 1100 : 100 rows
     Fetching rows 1100 → 1200 : 100 rows
     Fetching rows 1200 → 1300 : 100 rows
     Fetching rows 1300 → 1400 : 100 rows
     Fetching rows 1400 → 1500 : 100 rows
     Fetching rows 1500 → 1600 : 100 rows
     Fetching rows 1600 → 1700 : 100 rows
     Fetching rows 1700 → 1800 : 100 rows
     Fetching rows 1800 → 1900 : 100 rows
     Fetching rows 1900 → 2000 : 100 rows
     Fetching rows 2000 → 2100 : 100 rows
     Fetching rows 2100 → 2200 : 100 

BinderException: Binder Error: table all_goalie_games has 18 columns but 17 values were supplied

In [8]:
con.execute("DESCRIBE all_goalie_games").fetchdf()

,column_name,column_type,null,key,default,extra
0,game_id,BIGINT,YES,None,None,None
1,games_played,BIGINT,YES,None,None,None
2,games_started,BIGINT,YES,None,None,None
3,goalie_name,VARCHAR,YES,None,None,None
4,player_id,BIGINT,YES,None,None,None
5,tri_code,VARCHAR,YES,None,None,None
6,toi,BIGINT,YES,None,None,None
7,wins,BIGINT,YES,None,None,None
8,losses,BIGINT,YES,None,None,None
9,ot_losses,BIGINT,YES,None,None,None


In [13]:
goalie_games.columns

Index(['game_id', 'gamesPlayed', 'gamesStarted', 'goalieFullName', 'playerId',
       'teamAbbrev', 'timeOnIce', 'wins', 'losses', 'otLosses',
       'goalsAgainstAverage', 'saves', 'shutouts', 'savePct', 'shotsAgainst',
       'goalsAgainst', 'season_id'],
      dtype='object')

In [ ]:
goalie_games = goalie_games.rename(columns={'gamesPlayed':'games_played', 'gamesStarted': 'games_started', 'goalieFullName':'goalie_name', 'playerId':'player_id',
       'teamAbbrev':'tri_code', 'timeOnIce':'toi', 'otLosses':'ot_losses',
       'goalsAgainstAverage':'gaa', 'savePct':'save_pct', 'shotsAgainst':'shots_against',
       'goalsAgainst':'goals_against'})

,assists,gameDate,gameId,gamesPlayed,gamesStarted,goalieFullName,goals,goalsAgainst,goalsAgainstAverage,homeRoad,lastName,losses,opponentTeamAbbrev,otLosses,penaltyMinutes,playerId,points,savePct,saves,shootsCatches,shotsAgainst,shutouts,teamAbbrev,ties,timeOnIce,wins,season
0,0,2000-11-11,2000020240,1,1,Ed Belfour,0,0,0.0,H,Belfour,0,MTL,None,0,8445386,0,1.0,24,L,24,1,DAL,0,3600,1,20002001
1,0,2000-11-01,2000020172,1,1,Ed Belfour,0,0,0.0,H,Belfour,0,CBJ,None,0,8445386,0,1.0,14,L,14,1,DAL,0,3600,1,20002001
2,0,2000-12-10,2000020437,1,1,Ed Belfour,0,0,0.0,R,Belfour,0,ANA,None,0,8445386,0,1.0,29,L,29,1,DAL,0,3600,1,20002001
3,0,2001-03-21,2000021098,1,1,Ed Belfour,0,0,0.0,H,Belfour,0,ANA,None,0,8445386,0,1.0,17,L,17,1,DAL,0,3600,1,20002001
4,0,2000-10-14,2000020058,1,1,Ed Belfour,0,0,0.0,H,Belfour,0,WSH,None,0,8445386,0,1.0,12,L,12,1,DAL,0,3574,1,20002001


In [ ]:

# Create table if it doesn't exist
con.execute("""
    CREATE TABLE IF NOT EXISTS all_goalie_games AS 
    SELECT * FROM goalie_games
""")


# Insert new records
con.execute("""
    INSERT INTO all_goalie_games
    SELECT * FROM goalie_games
""")



In [ ]:
con.execute("""
select *
from all_goalie_games
order by game_id

""").fetch_df().head()


In [14]:

con.close()
print("DuckDB connection closed.")

DuckDB connection closed.


In [4]:
goalie_games_ = final_df[[ 
    'gameId', 'gamesPlayed', 'gamesStarted', 'goalieFullName', 'playerId',
    'teamAbbrev', 'timeOnIce', 'wins', 'losses', 'otLosses',
    'goalsAgainstAverage', 'saves', 'shutouts', 'savePct',
    'shotsAgainst', 'goalsAgainst', 'season'
]]
goalie_games_.head()


,gameId,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst,season
0,2019020796,1,1,Henrik Lundqvist,8468685,NYR,3600,1,0,0,0.0,33,1,1.0,33,0,20192020
1,2019020805,1,1,Curtis McElhinney,8470147,TBL,3583,1,0,0,0.0,30,1,1.0,30,0,20192020
2,2019020905,1,1,Marc-Andre Fleury,8470594,VGK,3600,1,0,0,0.0,19,1,1.0,19,0,20192020
3,2019020118,1,1,Marc-Andre Fleury,8470594,VGK,3575,1,0,0,0.0,29,1,1.0,29,0,20192020
4,2019020315,1,1,Marc-Andre Fleury,8470594,VGK,3598,1,0,0,0.0,34,1,1.0,34,0,20192020


In [5]:
all_goalies = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Goalies\all_goalie_games.csv")

goalie_games = pd.concat([goalie_games_,all_goalies])


In [ ]:

goalie_games.tail()

,gameId,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst,season
17392,2025020117,1,1,Joel Hofer,8480981,STL,477,0,0,0,22.64150,3,0,0.50000,6,3,20252026
17393,2025020275,1,1,Alex Lyon,8479312,BUF,370,0,0,0,19.45945,1,0,0.33333,3,2,20252026
17394,2025020341,1,1,Logan Thompson,8480313,WSH,1099,0,1,0,13.10282,2,0,0.33333,6,4,20252026
17395,2025020362,1,1,Dustin Wolf,8481692,CGY,352,0,1,0,30.68181,1,0,0.25000,4,3,20252026
17396,2025020259,1,0,Pyotr Kochetkov,8481611,CAR,216,0,0,0,0.00000,0,0,NaN,0,0,20252026


## Organize Goalie Stats

- Prep if appending new data to the all_goalie_data


In [20]:

goalies_25 = pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Goalies\goalie_stats_25.csv")

# goalie_games = goalie_games[[ 'gameId', 'gamesPlayed', 'gamesStarted', 'goalieFullName', 'playerId', 'teamAbbrev', 'timeOnIce', 'wins', 'losses', 'otLosses', 'goalsAgainstAverage', 'saves', 'shutouts', 'savePct', 
#                    'shotsAgainst', 'goalsAgainst']]
# goalie_games['season'] = 20252026
goalies_25.head()

,gameId,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst,season
0,2025020163,1,1,Jonathan Quick,8471734,NYR,3600,1,0,0,0.0,23,1,1.0,23,0,20252026
1,2025020212,1,1,Darcy Kuemper,8475311,LAK,3575,1,0,0,0.0,23,1,1.0,23,0,20252026
2,2025020130,1,1,Sergei Bobrovsky,8475683,FLA,3588,1,0,0,0.0,17,1,1.0,17,0,20252026
3,2025020323,1,1,Sergei Bobrovsky,8475683,FLA,3583,1,0,0,0.0,32,1,1.0,32,0,20252026
4,2025020350,1,1,Scott Wedgewood,8475809,COL,3600,1,0,0,0.0,22,1,1.0,22,0,20252026


In [21]:

goalie_games.tail()

,gameId,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst,season
17392,2025020117,1,1,Joel Hofer,8480981,STL,477,0,0,0,22.64150,3,0,0.50000,6,3,20252026
17393,2025020275,1,1,Alex Lyon,8479312,BUF,370,0,0,0,19.45945,1,0,0.33333,3,2,20252026
17394,2025020341,1,1,Logan Thompson,8480313,WSH,1099,0,1,0,13.10282,2,0,0.33333,6,4,20252026
17395,2025020362,1,1,Dustin Wolf,8481692,CGY,352,0,1,0,30.68181,1,0,0.25000,4,3,20252026
17396,2025020259,1,0,Pyotr Kochetkov,8481611,CAR,216,0,0,0,0.00000,0,0,NaN,0,0,20252026


In [22]:
history_goalie_games= pd.read_csv(r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Goalies\all_goalie_games.csv")
history_goalie_games.tail()

,gameId,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst,season
17392,2025020117,1,1,Joel Hofer,8480981,STL,477,0,0,0,22.64150,3,0,0.50000,6,3,20252026
17393,2025020275,1,1,Alex Lyon,8479312,BUF,370,0,0,0,19.45945,1,0,0.33333,3,2,20252026
17394,2025020341,1,1,Logan Thompson,8480313,WSH,1099,0,1,0,13.10282,2,0,0.33333,6,4,20252026
17395,2025020362,1,1,Dustin Wolf,8481692,CGY,352,0,1,0,30.68181,1,0,0.25000,4,3,20252026
17396,2025020259,1,0,Pyotr Kochetkov,8481611,CAR,216,0,0,0,0.00000,0,0,NaN,0,0,20252026


In [24]:
expected_cols =['gameId', 'gamesPlayed', 'gamesStarted', 'goalieFullName', 'playerId',
       'teamAbbrev', 'timeOnIce', 'wins', 'losses', 'otLosses',
       'goalsAgainstAverage', 'saves', 'shutouts', 'savePct', 'shotsAgainst',
       'goalsAgainst', 'season']

dfs = [history_goalie_games, goalie_games]

aligned_dfs = [df.reindex(columns=expected_cols) for df in dfs]
all_goalie_games = pd.concat(aligned_dfs, ignore_index=True)
all_goalie_games.head()

,gameId,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst,season
0,2017020289,1,1,Roberto Luongo,8466141,FLA,3600,1,0,0,0.0,35,1,1.0,35,0,20172018
1,2017021038,1,1,Roberto Luongo,8466141,FLA,3600,1,0,0,0.0,40,1,1.0,40,0,20172018
2,2017021118,1,1,Roberto Luongo,8466141,FLA,3585,1,0,0,0.0,28,1,1.0,28,0,20172018
3,2017020252,1,0,Craig Anderson,8467950,OTT,1764,1,0,0,0.0,8,0,1.0,8,0,20172018
4,2017020061,1,1,Craig Anderson,8467950,OTT,3592,1,0,0,0.0,25,1,1.0,25,0,20172018


## Export Complete Data Set

In [25]:

# Define export path
export_path = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\Goalies\all_hist_goalie_games.csv"

# Ensure the folder exists before writing the file
os.makedirs(os.path.dirname(export_path), exist_ok=True)

# # Export the data to CSV
all_goalie_games.to_csv(export_path, index=False)



In [31]:
all_goalie_games.tail()

,gameId,gamesPlayed,gamesStarted,goalieFullName,playerId,teamAbbrev,timeOnIce,wins,losses,otLosses,goalsAgainstAverage,saves,shutouts,savePct,shotsAgainst,goalsAgainst,season
34789,2025020117,1,1,Joel Hofer,8480981,STL,477,0,0,0,22.64150,3,0,0.50000,6,3,20252026
34790,2025020275,1,1,Alex Lyon,8479312,BUF,370,0,0,0,19.45945,1,0,0.33333,3,2,20252026
34791,2025020341,1,1,Logan Thompson,8480313,WSH,1099,0,1,0,13.10282,2,0,0.33333,6,4,20252026
34792,2025020362,1,1,Dustin Wolf,8481692,CGY,352,0,1,0,30.68181,1,0,0.25000,4,3,20252026
34793,2025020259,1,0,Pyotr Kochetkov,8481611,CAR,216,0,0,0,0.00000,0,0,NaN,0,0,20252026


# Goalie Individual Stats - Complete Seasons and To Date

In [30]:
goalies_individual = all_goalie_games.groupby(['goalieFullName','season','teamAbbrev']).agg(
    i_savePct=pd.NamedAgg(column='savePct', aggfunc=lambda x: (x * all_goalie_games.loc[x.index, 'shotsAgainst']).sum() / all_goalie_games.loc[x.index, 'shotsAgainst'].sum()),
    i_GAA=pd.NamedAgg(column='goalsAgainstAverage', aggfunc=lambda x: (x * all_goalie_games.loc[x.index, 'timeOnIce']).sum() / all_goalie_games.loc[x.index, 'timeOnIce'].sum()),
    i_gamesPlayed=('gamesPlayed', 'sum'),
    i_gamesStarted=('gamesStarted', 'sum'),
    i_wins=('wins', 'sum'),
    i_losses = ('losses', 'sum'),
    i_otlosses = ('otLosses', 'sum'),
    i_shutouts=('shutouts', 'sum'),
    i_saves=('saves', 'sum'),
    i_shotsAgainst=('shotsAgainst', 'sum'),
    i_timeOnIce=("timeOnIce",'sum'),
).sort_values("teamAbbrev").reset_index()
goalies_individual.head()

C:\Users\kjchr\AppData\Local\Temp\ipykernel_25816\3321280619.py:2: RuntimeWarning: invalid value encountered in scalar divide
  i_savePct=pd.NamedAgg(column='savePct', aggfunc=lambda x: (x * all_goalie_games.loc[x.index, 'shotsAgainst']).sum() / all_goalie_games.loc[x.index, 'shotsAgainst'].sum()),


,goalieFullName,season,teamAbbrev,i_savePct,i_GAA,i_gamesPlayed,i_gamesStarted,i_wins,i_losses,i_otlosses,i_shutouts,i_saves,i_shotsAgainst,i_timeOnIce
0,John Gibson,20172018,ANA,0.925744,2.432560,60,60,31,18,7,4,1733,1872,205709
1,Ryan Miller,20182019,ANA,0.911761,2.759738,20,17,8,7,2,1,527,578,66528
2,Ryan Miller,20172018,ANA,0.928278,2.349866,28,21,12,6,6,4,686,739,81196
3,James Reimer,20242025,ANA,0.863632,4.500000,2,2,0,2,0,0,57,66,7200
4,Ryan Miller,20182019,ANA,0.911761,2.759738,20,17,8,7,2,1,527,578,66528
